# Model test: which translator actually produces Akkadian?

**What this notebook does:** runs 3 complaint lines through each candidate model, translates them back into English, converts the Akkadian into cuneiform with CuneiScribe, and puts everything in one comparison table. Then it renders tablets from the model you pick.

**How to run:** `Runtime → Run all`. A GPU makes it faster but isn't required (`Runtime → Change runtime type → T4 GPU`).

**Candidates**
- `praeclarum/cuneiform`: established Akkadian ↔ English T5 model
- `notninja/byt5-base-akkadian`: unverified, little documentation
- `google/byt5-small`: **control**. This is CuneiScribe's built-in placeholder and has never been trained on Akkadian, so it *should* produce garbage. If it looks as good as the others, the test isn't telling them apart.

In [ ]:
# STEP 1: install
!pip -q install cuneiscribe sentencepiece protobuf
!apt-get -qq install -y fonts-noto-core > /dev/null   # provides NotoSansCuneiform for the tablet renderer

In [ ]:
# STEP 1 TEST: smoke test (no model needed)
from cuneiscribe import CuneiformConverter, AkkadianTranslator, CuneiScribe, classify, validate
conv = CuneiformConverter()
out = conv.to_cuneiform("LUGAL dan-nu")
print(out)
assert out == "𒈗 𒆗𒉡", "FAIL: converter output doesn't match the documented example"
print("PASS: CuneiScribe installed, sign table loaded:", conv.num_signs, "signs")

## Test lines
Placeholders in CFPB style. Swap in real CFPB lines once Step 3 (data) is done.

In [ ]:
TEST_LINES = [
    "I have called them XXXX times and nobody will help me.",
    "They charged me an overdraft fee even though I had money in my account.",
    "Why am I being treated with such contempt after paying on time for years?",
]

CANDIDATES = [
    "praeclarum/cuneiform",
    "notninja/byt5-base-akkadian",
    "google/byt5-small",   # control: should fail
]

In [ ]:
# STEP 2 TEST: translate with every candidate
import re, gc, time, torch

def coverage(cuneiform):
    # CuneiScribe writes syllables it can't map as [syllable]
    tokens = re.findall(r"\[[^\]]+\]|[\U00012000-\U0001254F]", cuneiform)
    if not tokens:
        return 0.0, 0
    unmapped = sum(1 for t in tokens if t.startswith("["))
    return round(100 * (1 - unmapped / len(tokens)), 1), unmapped

results = []
for name in CANDIDATES:
    print(f"\n=== {name} ===")
    t0 = time.time()
    try:
        tr = AkkadianTranslator(name)
        for line in TEST_LINES:
            ak = tr.to_akkadian(line, max_length=200)
            back = tr.to_english(ak, max_length=200)
            v = validate(ak, line)
            cu = conv.to_cuneiform(ak)
            cov, unmapped = coverage(cu)
            results.append(dict(model=name, english=line, akkadian=ak, back_translation=back,
                                cuneiform=cu, sign_coverage_pct=cov, unmapped=unmapped,
                                validator_score=round(v.score, 2), validator=v.suggestion,
                                issues="; ".join(v.issues)))
            print(f"  EN : {line}\n  AK : {ak}\n  CU : {cu}\n  BACK: {back}\n")
        del tr
    except Exception as e:
        print("  ERROR:", repr(e))
        results.append(dict(model=name, english="(model failed to load/run)", akkadian=repr(e)))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  ({time.time()-t0:.0f}s)")

In [ ]:
# Comparison table + CSV log
import pandas as pd
pd.set_option("display.max_colwidth", None)
df = pd.DataFrame(results)
df.to_csv("model_test_results.csv", index=False)
df[["model","english","akkadian","back_translation","sign_coverage_pct","validator"]]

## How to judge the table (pass criteria)
A model **passes** if all of these are true:
1. **Akkadian looks like Akkadian:** hyphenated syllables like `a-na be-li2-ia`, not English words or random letters.
2. **Sign coverage ≥ 90%:** nearly all syllables become real cuneiform signs.
3. **Back-translation keeps the frustration:** details like "overdraft fee" can disappear (that's your argument), but the grievance should survive.
4. **It clearly beats the control** (`google/byt5-small`).

If no model passes, stop here and tell me what you see before going further.

In [ ]:
# STEP 6 PREVIEW: render tablets from the winning model
BEST = "praeclarum/cuneiform"   # ← change to whichever model passed

from IPython.display import Image, display
cs = CuneiScribe(model_path=BEST)
for i, line in enumerate(TEST_LINES, 1):
    r = cs.craft(line, format="png", force_render=True, output_path=f"tablet_{i}.png")
    print(f"{i}. verdict={r.suggestion}  confidence={r.confidence:.2f}")
    print("   warnings:", r.warnings)
    display(Image(f"tablet_{i}.png"))

In [ ]:
# Download the CSV and tablets
!zip -q model_test_outputs.zip model_test_results.csv tablet_*.png
from google.colab import files
files.download("model_test_outputs.zip")